### Inspect scraped data_files

In [26]:
import pandas as pd
from itertools import combinations

In [3]:
logboek_df = pd.read_csv('logboek.csv', sep=';')
countries = logboek_df["land"].tolist()
leagues = logboek_df["competitienaam"].tolist()
seasons = logboek_df["jaar"].tolist()

In [4]:
print(countries)

['england', 'mexico', 'japan', 'iran', 'south-korea', 'australia', 'morocco', 'senegal', 'nigeria', 'algeria', 'egypt', 'ivory-coast', 'usa', 'canada', 'panama', 'argentina', 'brazil', 'colombia', 'uruguay', 'ecuador', 'paraguay', 'austria', 'belgium', 'cyprus', 'czech-republic', 'france', 'germany', 'hungary', 'ireland', 'italy', 'luxembourg', 'malta', 'netherlands', 'northern-ireland', 'norway', 'poland', 'portugal', 'romania', 'russia', 'scotland', 'slovakia', 'spain', 'switzerland', 'turkey', 'wales', 'croatia', 'denmark', 'ukraine', 'serbia', 'sweden']


### checked up untill index 0

In [5]:
def filter_rounds(df):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str.startswith("Round ")].reset_index(drop=True)
    return return_df

In [6]:
def filter_playoffs(df, start_round_playoffs):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str[6:].astype(int) < start_round_playoffs].reset_index(drop=True)
    return return_df

In [55]:
def filter_on_date(df, start_date, keep_before = True):
    """ 
    Format start_date: "YYYY-MM-DD"
    """
    return_df = df.copy()
    start_date = pd.to_datetime(start_date)
    
    return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')
    if keep_before:
        return_df = return_df[(return_df["date"] < start_date)].reset_index(drop=True)
    else:
        return_df = return_df[(return_df["date"] >= start_date)].reset_index(drop=True)
    return return_df

In [8]:
def remove_round_prefix(df):
    return_df = df.copy()
    return_df["round"] = return_df["round"].apply(lambda x: x[6:])
    return return_df

In [9]:
def check_rounds(df):
    rounds = df["round"].unique().tolist()
    counted_rounds = df["round"].value_counts().to_dict()

    # check if all rounds appear the same amount of times
    rounds_appear_same_amount = len(set(counted_rounds.values())) == 1

    # check if rounds start with "Round"
    rounds_start_with_round = all(r.startswith("Round") for r in rounds)

    print(f"Rounds: {rounds}")
    print(f"Only normal rounds: {rounds_start_with_round}")
    print(f"All rounds appear same amount of times: {rounds_appear_same_amount}")

In [10]:
def check_teams(df):
    home_teams = df["team_home"].unique().tolist()
    away_teams = df["team_away"].unique().tolist()

    all_teams = set(home_teams) | set(away_teams)
    print(f"Teams: {all_teams}")
    print(f"Number of teams: {len(all_teams)}")

    # check if all teams appear in both home and away the same amount of times
    home_team_counts = df["team_home"].value_counts().to_dict()
    away_team_counts = df["team_away"].value_counts().to_dict()
    teams_appear_same_amount = all(home_team_counts.get(team, 0) == away_team_counts.get(team, 0) for team in all_teams)
    print(f"All teams appear same amount of times in home and away: {teams_appear_same_amount}")
    if teams_appear_same_amount:
        print(f"Each team appears {home_team_counts[home_teams[0]]} times in home and away")


In [29]:
def check_games(df):
    teams = sorted(set(df['team_home'].unique()) | set(df['team_away'].unique()))
    
    # Build symmetric n x n matrix
    matrix = pd.DataFrame(0, index=teams, columns=teams)
    
    for _, row in df.iterrows():
        h, a = row['team_home'], row['team_away']
        matrix.loc[h, a] += 1
        matrix.loc[a, h] += 1  # symmetric: home/away doesn't matter
    
    # Extract counts for all unique pairs (upper triangle)
    pair_counts = {
        (t1, t2): matrix.loc[t1, t2]
        for t1, t2 in combinations(teams, 2)
    }
    
    unique_counts = set(pair_counts.values())
    is_balanced = len(unique_counts) == 1 and 0 not in unique_counts
    
    print("=== Game Balance Check ===\n")
    print("Head-to-head matrix:")
    print(matrix.to_string())
    print()
    
    count_freq = {}
    for v in pair_counts.values():
        count_freq[v] = count_freq.get(v, 0) + 1
    
    if is_balanced:
        times = unique_counts.pop()
        print(f"Balanced: every pair plays exactly {times} time(s).")
    else:
        print(f"Not balanced. Distribution of matchup counts: {count_freq}")
        if 0 in unique_counts:
            missing = [f"{t1} vs {t2}" for (t1, t2), v in pair_counts.items() if v == 0]
            print(f"   Never played: {missing[:10]}{'...' if len(missing) > 10 else ''}")
        modal = max(set(pair_counts.values()), key=list(pair_counts.values()).count)
        odd = {f"{k[0]} vs {k[1]}": v for k, v in pair_counts.items() if v != modal}
        if odd:
            print(f"   Odd counts: {odd}")
    

In [ ]:
# Checked untill index 10, egypt
index = 10
country = countries[index]
league = leagues[index]
season = seasons[index]
print(country)
print(league)
print(season)
filename = f"{country}_{league}_{season}.csv"
df = pd.read_csv(f"./scraped_data/{filename}")
df.head()

egypt
premier-league
2024-2025


,round,date,team_home,team_away,full_text
0,Round 9,31.05.2025,Zamalek,Pharco,31.05.2025 | Zamalek | Pharco | 2 | 0
1,Round 9,29.05.2025,Al Ittihad,Enppi,29.05.2025 | Al Ittihad | Enppi | 1 | 1
2,Round 9,29.05.2025,El Gaish,Modern Sport,29.05.2025 | El Gaish | Modern Sport | 1 | 1
3,Round 9,29.05.2025,Ghazl El Mahallah,El Ismaily,29.05.2025 | Ghazl El Mahallah | El Ismaily | ...
4,Round 9,29.05.2025,ZED,Smouha,29.05.2025 | ZED | Smouha | 0 | 0


In [48]:
# First do the checks, then do the filtering and removing of round prefix if necessary

check_rounds(df)
check_teams(df)
check_games(df)

Rounds: ['Round 9', 'Round 8', 'Round 7', 'Round 6', 'Round 5', 'Round 4', 'Round 3', 'Round 2', 'Round 1', 'Round 17', 'Round 16', 'Round 15', 'Round 14', 'Round 13', 'Round 12', 'Round 11', 'Round 10']
Only normal rounds: True
All rounds appear same amount of times: False
Teams: {'El Gouna', 'Smouha', 'Al Masry', 'Modern Sport', 'Enppi', 'National Bank Egypt', 'ZED', 'El Ismaily', 'El Gaish', 'Awrd', 'Zamalek', 'Pyramids', 'Haras El Hodood', 'Ceramica Cleopatra', 'Ghazl El Mahallah', 'Petrojet', 'Al Ittihad', 'Al Ahly', 'Pharco'}
Number of teams: 19
All teams appear same amount of times in home and away: False
=== Game Balance Check ===

Head-to-head matrix:
                     Al Ahly  Al Ittihad  Al Masry  Awrd  Ceramica Cleopatra  El Gaish  El Gouna  El Ismaily  Enppi  Ghazl El Mahallah  Haras El Hodood  Modern Sport  National Bank Egypt  Petrojet  Pharco  Pyramids  Smouha  ZED  Zamalek
Al Ahly                    0           1         2     0                   2         1        

In [20]:
df_only_rounds = filter_rounds(df)

In [21]:
check_rounds(df_only_rounds)
check_teams(df_only_rounds)

Rounds: ['Round 30', 'Round 29', 'Round 28', 'Round 27', 'Round 26', 'Round 25', 'Round 24', 'Round 23', 'Round 22', 'Round 21', 'Round 20', 'Round 19', 'Round 18', 'Round 17', 'Round 16', 'Round 14', 'Round 15', 'Round 13', 'Round 12', 'Round 11', 'Round 10', 'Round 9', 'Round 8', 'Round 7', 'Round 6', 'Round 5', 'Round 3', 'Round 4', 'Round 2', 'Round 1']
Only normal rounds: True
All rounds appear same amount of times: True
Teams: {'Raja Casablanca', 'Moghreb Tetouan', 'Union Touarga', 'Maghreb Fez', 'COD Meknes', 'Chabab Mohammedia', 'Difaa El Jadidi', 'Berkane', 'Hassania Agadir', 'Jeunesse Sportive Soualem', 'Wydad AC', 'FUS Rabat', 'Olympique de Safi', 'Renaissance Zemamra', 'FAR Rabat', 'IR Tanger'}
Number of teams: 16
All teams appear same amount of times in home and away: True
Each team appears 15 times in home and away


In [56]:
df_only_rounds = filter_playoffs(df_only_rounds, 34)

In [63]:
df_only_rounds

,round,date,team_home,team_away,full_text
0,33,18.10.2025,Anyang,Gimcheon Sangmu,18.10.2025 | Anyang | Gimcheon Sangmu | 4 | 1
1,33,18.10.2025,Daegu,Gangwon,18.10.2025 | Daegu | Gangwon | 2 | 2
2,33,18.10.2025,Daejeon,Jeju SK,18.10.2025 | Daejeon | Jeju SK | 3 | 1
3,33,18.10.2025,Jeonbuk,Suwon FC,18.10.2025 | Jeonbuk | Suwon FC | 2 | 0
4,33,18.10.2025,Seoul,Pohang,18.10.2025 | Seoul | Pohang | 1 | 2
...,...,...,...,...,...
193,1,16.02.2025,Jeonbuk,Gimcheon Sangmu,16.02.2025 | Jeonbuk | Gimcheon Sangmu | 2 | 1
194,1,16.02.2025,Ulsan HD,Anyang,16.02.2025 | Ulsan HD | Anyang | 0 | 1
195,1,15.02.2025,Gwangju FC,Suwon FC,15.02.2025 | Gwangju FC | Suwon FC | 0 | 0
196,1,15.02.2025,Jeju SK,Seoul,15.02.2025 | Jeju SK | Seoul | 2 | 0


In [62]:
df_only_rounds[df_only_rounds["round"] == "Round 1"]

,round,date,team_home,team_away,full_text


In [ ]:
df_filtered = filter_on_date(df, "2025-03-11")

C:\Users\simcosyn\AppData\Local\Temp\ipykernel_36672\1965196199.py:8: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')


In [ ]:
df_filtered.head()

,round,date,team_home,team_away,full_text
0,Round 17,2025-03-05,Al Ahly,El Gaish,05.03.2025 | Al Ahly | El Gaish | 2 | 0
1,Round 17,2025-03-05,Petrojet,Ghazl El Mahallah,05.03.2025 | Petrojet | Ghazl El Mahallah | 2 | 1
2,Round 17,2025-03-05,Pyramids,Ceramica Cleopatra,05.03.2025 | Pyramids | Ceramica Cleopatra | 2...
3,Round 17,2025-03-04,Al Ittihad,Smouha,04.03.2025 | Al Ittihad | Smouha | 0 | 1
4,Round 17,2025-03-04,Enppi,Zamalek,04.03.2025 | Enppi | Zamalek | 0 | 3
...,...,...,...,...,...
148,Round 1,2024-11-01,Pyramids,Petrojet,01.11.2024 | Pyramids | Petrojet | 1 | 1
149,Round 1,2024-10-31,Modern Sport,Enppi,31.10.2024 | Modern Sport | Enppi | 0 | 0
150,Round 1,2024-10-31,El Gouna,ZED,31.10.2024 | El Gouna | ZED | 0 | 0
151,Round 1,2024-10-30,El Gaish,Al Masry,30.10.2024 | El Gaish | Al Masry | 0 | 2


In [ ]:
df_filtered = remove_round_prefix(df_filtered)

In [59]:
# write away cleaned df and adapt logboek
df_filtered.to_csv(f"./cleaned_data/{filename}", index=False)